<h1 style="color:#1f77b4;">SARIMAX BANXICO API - EXAM</h1>

 `Autoras:` 
 - Sarah Lucia Beltrán Gutiérrez
 - Aissa Berenice
 - Isabel Valladolid

 `Fecha:` 04/03/2026

 `Materia:` Non Linear Models

 `Profesor:` Pedro Martinez

---

<h2 style="color:#1f77b4;">Introduccion</h2>

En este proyecto se desarrolla un modelo `SARIMAX` para analizar y predecir el comportamiento del tipo de cambio **peso mexicano–dólar estadounidense** (FIX), utilizando datos oficiales obtenidos mediante la API del Banco de México. A partir de una serie temporal diaria, se busca evaluar la capacidad predictiva del modelo en el corto plazo, específicamente para el periodo del **5 al 13 de marzo de 2026**. El trabajo integra la descarga y procesamiento de datos, la justificación del modelo elegido y la evaluación de su desempeño mediante el error `MAPE`, con el fin de medir qué tan eficaz resulta `SARIMA` para anticipar movimientos del

---

In [20]:
# librerias
from dotenv import load_dotenv
import os
import requests
import plotly.express as px
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import grangercausalitytests
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_ccf
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.stattools import adfuller, acf, pacf

In [2]:
# token desde .env
load_dotenv()
TOKEN = os.getenv("BANXICO_TOKEN")

In [3]:
# llamamos api :p
url = "https://www.banxico.org.mx/SieAPIRest/service/v1/series/SF43718/datos"
headers = {"Bmx-Token": TOKEN}

response = requests.get(url, headers=headers)
response.raise_for_status()
data = response.json()

In [4]:
# df
datos = data["bmx"]["series"][0]["datos"]
df = pd.DataFrame(datos)   
df.head()

,fecha,dato
0,12/11/1991,3.0735
1,13/11/1991,3.0712
2,14/11/1991,3.0718
3,15/11/1991,3.0684
4,18/11/1991,3.0673


In [5]:
df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True)
df["dato"] = pd.to_numeric(df["dato"], errors="coerce")

# fecha como indice
df.set_index("fecha", inplace=True)
df.sort_index(inplace=True)

<h2 style="color:#1f77b4;">Visualizacion</h2>

In [6]:
fig = px.line(
    df,
    x=df.index,
    y="dato",
    title="Serie de Tiempo: Tipo de Cambio FIX (MXN/USD)"
)

fig.update_layout(
    xaxis_title="Fecha",
    yaxis_title="Tipo de cambio FIX",
)

fig.show()

En la gráfica anterior se muestra la serie de tiempo de la base de datos de Banxico desde 1995 hasta 2026, a continuación se realizara el "corte" para que la prediccion no se ensucie con datos super viejos. Se tomara desde enero 2016.

In [7]:
import holidays

In [8]:
df = df.loc['2015-01-01':]

In [9]:
# Creamos la columna llena de ceros (variable dummy)
df['Holiday_Dummy'] = 0
# Descargamos los holidays de USA de los años que hay en TSA
years = df.index.year.unique()
mx_holidays = holidays.MX(years=years)
us_holidays = holidays.US(years=years)


# Lógica: Marcamos con 1 si la fecha cae en una ventana ALGUNOS días
for date in df.index:
    # Ventana de días
    window = pd.date_range(start=date - pd.Timedelta(days=1),
                           end=date + pd.Timedelta(days=1))

    # Si algún día de la ventana es festivo, marcamos el día actual como 1
    if any(d in mx_holidays for d in window):
        df.loc[date, 'Holiday_Dummy'] = 1
    elif any(d in us_holidays for d in window):
        df.loc[date, 'Holiday_Dummy'] = 1  

In [10]:
df['Holiday_Dummy'].value_counts()


Holiday_Dummy
0    2493
1     316
Name: count, dtype: int64

In [11]:
fig = px.line(
    df,
    x=df.index,
    y="dato",
    title="Serie de Tiempo: Tipo de Cambio FIX (MXN/USD) – Datos Recientes"
)

fig.update_layout(
    xaxis_title="Fecha",
    yaxis_title="Tipo de cambio FIX",
)

fig.show()

<h2 style="color:#1f77b4;">Análisis para el (p,d,q) y (P,D,Q,s)</h2> 

In [12]:
def check_stationarity(series, title="Serie Original"):
    result = adfuller(series.dropna())
    print(f'ADF Test: {title}')
    print(f'Estadístico ADF: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    is_stationary = result[1] < 0.05
    print(f"¿Es estacionaria? {'SÍ' if is_stationary else 'NO'}\n")
    return is_stationary

# 1. Revisamos la serie original
check_stationarity(df['dato'], "Nivel Original")

# 2. Aplicamos Primera Diferencia (d=1)
df_diff = df['dato'].diff()

# 3. Revisamos la serie diferenciada
check_stationarity(df_diff, "Primera Diferencia (d=1)")

# Creamos una figura con 2 columnas (Subplots)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Serie Original (No Estacionaria)", "Serie Diferenciada (Estacionaria d=1)")
)

# Gráfico 1: Serie Original
fig.add_trace(
    go.Scatter(x=df.index, y=df['dato'], name='Original'),
    row=1, col=1
)

# Gráfico 2: Serie Diferenciada
fig.add_trace(
    go.Scatter(x=df_diff.index, y=df_diff, name='Diferenciada'),
    row=1, col=2
)

# Ajustes de diseño
fig.update_layout(
    title_text="Comparativa: Efecto de la Diferenciación",
    showlegend=False, # Ocultamos leyenda
    height=500
)

fig.show()

ADF Test: Nivel Original
Estadístico ADF: -3.0359
p-value: 0.0317
¿Es estacionaria? SÍ

ADF Test: Primera Diferencia (d=1)
Estadístico ADF: -50.0700
p-value: 0.0000
¿Es estacionaria? SÍ



In [13]:
# @title Graficamos ACF y PACF

ts_analysis = df_diff.dropna()

# Parámetros
lags = 30  # 30 días
alpha = 0.05 # Nivel de significancia del 95%

# Cálculo de valores ACF y PACF
acf_vals = acf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]
pacf_vals = pacf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]

# Colocamos manualmente el intervalo de confianza para plotly
n = len(ts_analysis)
conf_interval = 1.96 / np.sqrt(n)


fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Función de Autocorrelación (ACF) - Determina MA(q)",
                                    "Autocorrelación Parcial (PACF) - Determina AR(p)"),
                    vertical_spacing=0.15)

# ACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=acf_vals,
    name='ACF', marker_color='rgb(31, 119, 180)', showlegend=False
), row=1, col=1)

# Intervalos de confianza (Sombreado)
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=1, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=1, col=1)

# PACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=pacf_vals,
    name='PACF', marker_color='rgb(255, 127, 14)', showlegend=False
), row=2, col=1)

# Intervalos de confianza
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=2, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=2, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=2, col=1)


fig.update_layout(
    title='<b>Diagnóstico de Estructura: ACF y PACF</b><br><sup>Serie Diferenciada</sup>',
    template='plotly_white',
    height=700,
    bargap=0.8 # Barras delgadas estilo lollipop
)

# Resaltar lags estacionales (7, 14, 21, 28) con líneas verticales rojas tenues
for i in [7, 14, 21, 28]:
    fig.add_vline(x=i, line_width=1, line_dash="dot", line_color="red", opacity=0.5)

fig.show()

Cuando la serie sale asi es porque el tipo de cambio esta super parecido, como rl tipo de cambio no cambia mucho entonces sale asi porque cambia decimales :P

In [27]:
# @title Realizamos modelo y graficamos
TEST_DAYS = 30

# Serie objetivo (FIX)
ts = df["dato"].astype(float).dropna()

# Exógena (dummy de festivos) alineada al índice de la serie
exog = df[["Holiday_Dummy"]].astype(float).reindex(ts.index)

train = ts.iloc[:-TEST_DAYS]
test = ts.iloc[-TEST_DAYS:]

exog_train = exog.loc[train.index]
exog_test = exog.loc[test.index]

# SARIMAX (SARIMA + exógena)
model = SARIMAX(
    train,
    exog=exog_train,
    order=(3, 1, 1),
    seasonal_order=(0, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

results = model.fit(disp=False)

# Predecimos n pasos hacia el futuro (donde n = tamaño del test)
forecast_object = results.get_forecast(steps=len(test), exog=exog_test)
forecast_vals = forecast_object.predicted_mean
conf_int = forecast_object.conf_int(alpha=0.05)  # Intervalo del 95%

# Alineación de índices para graficar bonito
forecast_vals.index = test.index
conf_int.index = test.index

# Metricas de error (sin sklearn)
residuals = (test - forecast_vals).dropna()
rmse = float(np.sqrt(np.mean(np.square(residuals))))
mae = float(np.mean(np.abs(residuals)))

denom = test.replace(0, np.nan)
mape = float(np.mean(np.abs((test - forecast_vals) / denom)))

print(f"\n--- Errores del modelo ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2%}")
print(f"MAE: {mae:.4f}")

print(results.summary())

# Grafica
fig = go.Figure()

# Train
fig.add_trace(go.Scatter(
    x=train.index, y=train,
    mode='lines',
    name='Train',
    line=dict(color='rgba(100, 100, 100, 0.6)', width=1.5)
))

# Test
fig.add_trace(go.Scatter(
    x=test.index, y=test,
    name='Test',
    line=dict(color='#1f77b4', width=3),
    marker=dict(size=6)
))

# Forecast
fig.add_trace(go.Scatter(
    x=test.index, y=forecast_vals,
    name='SARIMAX',
    line=dict(color='#ff7f0e', width=3, dash='dot')
))

# Intervalos de Confianza
fig.add_trace(go.Scatter(
    x=conf_int.index, y=conf_int.iloc[:, 0],
    mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'
))
fig.add_trace(go.Scatter(
    x=conf_int.index, y=conf_int.iloc[:, 1],
    mode='lines', line=dict(width=0), fill='tonexty',
    fillcolor='rgba(255, 127, 14, 0.2)',
    name='Int. Confianza 95%', hoverinfo='skip'
))

# Titulos
fig.update_layout(
    title=f'<b>Modelo SARIMAX: Pronóstico FIX (MXN/USD)</b><br><sup>Test: últimos {TEST_DAYS} días</sup>',
    xaxis_title='Fecha',
    yaxis_title='Tipo de cambio FIX',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.8)'),
    hovermode="x unified"
)

fig.show()

/Users/isabelvalladolid/Documents/ModelosNoLineales/SARIMAX-API-Banxico/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/Users/isabelvalladolid/Documents/ModelosNoLineales/SARIMAX-API-Banxico/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.




--- Errores del modelo ---
RMSE: 0.3576
MAPE: 1.97%
MAE: 0.3391
                                     SARIMAX Results                                     
Dep. Variable:                              dato   No. Observations:                 2779
Model:             SARIMAX(3, 1, 1)x(0, 1, 1, 7)   Log Likelihood                1275.277
Date:                           Wed, 04 Mar 2026   AIC                          -2536.555
Time:                                   18:46:42   BIC                          -2495.089
Sample:                                        0   HQIC                         -2521.577
                                          - 2779                                         
Covariance Type:                             opg                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Holiday_Dummy    -0.0084      0.008     -1.000     

/Users/isabelvalladolid/Documents/ModelosNoLineales/SARIMAX-API-Banxico/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

/Users/isabelvalladolid/Documents/ModelosNoLineales/SARIMAX-API-Banxico/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.



In [28]:
# @title SARIMA sin exógena (comparación)
TEST_DAYS = 30

# Serie objetivo (FIX)
ts = df["dato"].astype(float).dropna()

train = ts.iloc[:-TEST_DAYS]
test = ts.iloc[-TEST_DAYS:]

# SARIMA (implementado con SARIMAX pero sin exog)
model = SARIMAX(
    train,
    order=(3, 1, 1),
    seasonal_order=(0, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

results = model.fit(disp=False)

# Predecimos n pasos hacia el futuro (donde n = tamaño del test)
forecast_object = results.get_forecast(steps=len(test))
forecast_vals = forecast_object.predicted_mean
conf_int = forecast_object.conf_int(alpha=0.05)  # Intervalo del 95%

# Alineación de índices para graficar bonito
forecast_vals.index = test.index
conf_int.index = test.index

# Metricas de error (usando sklearn, como en tu ejemplo)
rmse = np.sqrt(mean_squared_error(test, forecast_vals))
mape = mean_absolute_percentage_error(test, forecast_vals)
mae = mean_absolute_error(test, forecast_vals)

print(f"\n--- Errores del modelo (SARIMA sin exógena) ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2%}")
print(f"MAE: {mae:.4f}")

print(results.summary())

# Grafica
fig = go.Figure()

# Train
fig.add_trace(go.Scatter(
    x=train.index, y=train,
    mode='lines',
    name='Train',
    line=dict(color='rgba(100, 100, 100, 0.6)', width=1.5)
))

# Test
fig.add_trace(go.Scatter(
    x=test.index, y=test,
    name='Test',
    line=dict(color='#1f77b4', width=3),
    marker=dict(size=6)
))

# Forecast
fig.add_trace(go.Scatter(
    x=test.index, y=forecast_vals,
    name='SARIMA (sin exog)',
    line=dict(color='#2ca02c', width=3, dash='dot')
))

# Intervalos de Confianza
fig.add_trace(go.Scatter(
    x=conf_int.index, y=conf_int.iloc[:, 0],
    mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'
))
fig.add_trace(go.Scatter(
    x=conf_int.index, y=conf_int.iloc[:, 1],
    mode='lines', line=dict(width=0), fill='tonexty',
    fillcolor='rgba(44, 160, 44, 0.2)',
    name='Int. Confianza 95%', hoverinfo='skip'
))

# Titulos
fig.update_layout(
    title=f'<b>Modelo SARIMA (sin exógena): Pronóstico FIX (MXN/USD)</b><br><sup>Test: últimos {TEST_DAYS} días</sup>',
    xaxis_title='Fecha',
    yaxis_title='Tipo de cambio FIX',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.8)'),
    hovermode="x unified"
)

fig.show()

/Users/isabelvalladolid/Documents/ModelosNoLineales/SARIMAX-API-Banxico/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/Users/isabelvalladolid/Documents/ModelosNoLineales/SARIMAX-API-Banxico/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.




--- Errores del modelo (SARIMA sin exógena) ---
RMSE: 0.3508
MAPE: 1.93%
MAE: 0.3321
                                     SARIMAX Results                                     
Dep. Variable:                              dato   No. Observations:                 2779
Model:             SARIMAX(3, 1, 1)x(0, 1, 1, 7)   Log Likelihood                1274.758
Date:                           Wed, 04 Mar 2026   AIC                          -2537.515
Time:                                   18:46:49   BIC                          -2501.973
Sample:                                        0   HQIC                         -2524.677
                                          - 2779                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.0281      0.772    

/Users/isabelvalladolid/Documents/ModelosNoLineales/SARIMAX-API-Banxico/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

/Users/isabelvalladolid/Documents/ModelosNoLineales/SARIMAX-API-Banxico/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.

